## Recalculating Harmonic Content

The Catmull-Rom interpolated waveforms have their harmonic contents stored currently for plotting. But the GP models generate waves with different harmonic contents. Reusing the previous energy %s is not responsible. So new FFT needs to be computed

We can calculate the average harmonic content at each component frequency for UMAP coloring.

In [1]:
import json
import pandas as pd

results = []
with open("../Processed_Data/Sampled_dat.json", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        results.append(json.loads(line))
results = [r for r in results if r.get("success")]
fit_df = pd.DataFrame(results)

In [7]:
VST = pd.read_csv("../Processed_Data/VST.csv")

In [21]:
df = fit_df[['GeneID', 'lengthscale', 'variance', 'noise', 'mean']].merge(VST, "left", on="GeneID").set_index("GeneID")

In [45]:
import gpflow
import numpy as np

PERIOD = 24.0
N = 256
grid = (np.arange(N) * PERIOD / N)[:, None]
sample_hours = np.arange(2, 50, 3, dtype=float)[:, None]

def gpflow_posterior(expr, variance, lengthscale, noise, mean):
    """Rebuild one gene's fitted GPR from the stored hyperparameters (no optimisation) and
    return the latent posterior on the grid: μ (256,), Σ (256, 256)."""
    kernel = gpflow.kernels.Periodic(
        gpflow.kernels.SquaredExponential(variance=variance, lengthscales=lengthscale),
        period=PERIOD,
    )
    model = gpflow.models.GPR(
        data=(sample_hours, np.asarray(expr, dtype=float)[:, None]),
        kernel=kernel,
        mean_function=gpflow.mean_functions.Constant(mean),
        noise_variance=noise,
    )
    f_mean, f_cov = model.predict_f(grid, full_cov=True)    # shapes (256, 1) and (1, 256, 256)
    return f_mean.numpy()[:, 0], f_cov.numpy()[0]

In [46]:
test_row = df.iloc[1,:]
arr = np.array(test_row.loc[test_row.index.str.startswith("LD_ZT")])
mu, Sigma = gpflow_posterior(arr, test_row.variance, test_row.lengthscale, test_row.noise, test_row['mean'])

In [47]:
K_MAX = 8
ks = np.arange(K_MAX + 1)                                       # 0..8, k = 0 is the DC offset
F_K = np.exp(-2j * np.pi * np.outer(ks, np.arange(N)) / N)      # rows F_k, shape (9, N)
def expected_power(mu, Sigma):
    mean_power = np.abs(np.fft.fft(mu)[ks]) ** 2                # |μ̂ₖ|²   (Option 1's term)
    wobble_power = np.einsum("ki,ij,kj->k", F_K, Sigma, F_K.conj()).real   # σₖ²
    return mean_power, wobble_power                             # E|x̂ₖ|² = sum of the two

In [48]:
# Random sampling THEN fft, result should match the exact method of expected_power
rng = np.random.default_rng(0)
L = np.linalg.cholesky(Sigma + 1e-9 * np.eye(N))
draws = mu + (L @ rng.standard_normal((N, 20_000))).T
mc_power = (np.abs(np.fft.fft(draws, axis=1)[:, ks]) ** 2).mean(axis=0)

mean_power, wobble_power = expected_power(mu, Sigma)
print(np.c_[ks, mc_power - (mean_power + wobble_power)])

[[ 0.00000000e+00 -7.79537596e+01]
 [ 1.00000000e+00 -4.16215132e+00]
 [ 2.00000000e+00  8.81303405e-01]
 [ 3.00000000e+00  8.63803261e-02]
 [ 4.00000000e+00  1.56149046e-01]
 [ 5.00000000e+00  9.41485097e-03]
 [ 6.00000000e+00 -5.34645894e-04]
 [ 7.00000000e+00  1.20040901e-04]
 [ 8.00000000e+00  2.66685300e-06]]


In [96]:
def calculate_powers(row):
    arr = row.loc[row.index.str.startswith("LD_ZT")].to_numpy()
    mu, Sigma = gpflow_posterior(arr, row['variance'], row['lengthscale'], row['noise'], row['mean'])
    mean_power, wobble_power = expected_power(mu, Sigma)
    mean_power = mean_power[1:]
    wobble_power = wobble_power[1:]
    return mean_power, wobble_power

Per row calculation too slow, vectorize:

In [97]:
from scipy.special import ive

def harmonic_powers(expr, variance, lengthscale, noise, mean,
                    hours=np.arange(2, 50, 3, dtype=float), n_grid=256, k_max=8, period=24.0):

    expr = np.atleast_2d(expr)
    G = expr.shape[0]
    variance, lengthscale, noise, mean = (np.asarray(v, float).reshape(G) for v in (variance, lengthscale, noise, mean))
    ks = np.arange(1, k_max + 1)

    # Fourier coefficients of each gene's kernel: c_k = var · e^{-x} I_k(x), x = 1/(4ℓ²)   (G, K)
    c = variance[:, None] * ive(ks[None, :], 1.0 / (4 * lengthscale[:, None] ** 2))

    # Covariance of the 16 measurements: kernel + noise   (G, 16, 16)
    s = np.sin(np.pi * (hours[:, None] - hours[None, :]) / period)
    K = variance[:, None, None] * np.exp(-0.5 * s[None] ** 2 / lengthscale[:, None, None] ** 2)
    K += noise[:, None, None] * np.eye(len(hours))

    E = np.exp(-2j * np.pi * np.outer(ks, hours) / period)          # harmonic k's phasor at each sample time (K, 16)

    alpha = np.linalg.solve(K, (expr - mean[:, None])[..., None])[..., 0]            # K⁻¹ (y − mean)   (G, 16)
    Kinv_E = np.linalg.solve(K.astype(complex), np.broadcast_to(E.conj().T, (G, *E.T.shape)))   # K⁻¹ ēₖ   (G, 16, K)
    quad = np.einsum("km,gmk->gk", E, Kinv_E).real                  # eₖᴴ K⁻¹ eₖ   (G, K)

    mean_power = (n_grid * c * np.abs(alpha @ E.T)) ** 2            # |μ̂ₖ|²
    wobble_power = n_grid ** 2 * (c - c ** 2 * quad)                # σₖ² = prior − what the data explains
    return mean_power, wobble_power

In [98]:
cols = df.columns[df.columns.str.startswith("LD_ZT")]
mean_power, wobble_power = harmonic_powers(
    df[cols].to_numpy(float), df.variance, df.lengthscale, df.noise, df["mean"])

Spot check

In [99]:
results = df.iloc[0:5,:].apply(calculate_powers, axis=1)

In [101]:
# results: Series of (mean_power, wobble_power, overall_pct) tuples, indexed by GeneID
ref_mean   = np.stack(results.map(lambda r: r[0]).to_numpy())   # (G, 8)
ref_wobble = np.stack(results.map(lambda r: r[1]).to_numpy())   # (G, 8)

def max_rel_err(a, b):
    return np.max(np.abs(a - b) / np.maximum(np.abs(b), 1e-12))

print("mean_power   max rel err:", max_rel_err(mean_power[0:5], ref_mean))
print("wobble_power max rel err:", max_rel_err(wobble_power[0:5], ref_wobble))

# # Or one-liners that raise if they disagree:
# np.testing.assert_allclose(mean_power[0:5], ref_mean, rtol=1e-6)
# np.testing.assert_allclose(wobble_power[0:5], ref_wobble, rtol=1e-6)

mean_power   max rel err: 4.4391990776128686e-11
wobble_power max rel err: 1.665391756193193e-11


## Update gp_params.bin

In [105]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

# Client draws the posterior. Each gene is four float32 hyperparameters,
# in the same order as `meta` below. Expression stays in VST.bin.
DATA_COLS = ["lengthscale", "variance", "noise", "mean", "relative_std", "shape_sensitivity"]

results = []
with open("../Processed_Data/Sampled_dat.json", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        results.append(json.loads(line))

results = [r for r in results if r.get("success")]
fit_df = pd.DataFrame(results)
missing = [k for k in DATA_COLS if k not in fit_df.columns]
if missing:
    raise KeyError(f"Sampled_dat.json is missing {missing}; re-run the fit cell")

mean_power_series = [f"mean_power_{i+1}" for i in range(8)]
wobble_power_series = [f"wobble_power_{i+1}" for i in range(8)]

fit_df[mean_power_series] = mean_power
fit_df[wobble_power_series] = wobble_power

DATA_COLS += (mean_power_series + wobble_power_series)

params = fit_df[DATA_COLS].to_numpy(dtype=np.float32)
assert params.shape == (len(fit_df), len(DATA_COLS))
params.tofile("../Processed_Data/gp_params.bin")

meta_out = {
    "n": int(len(fit_df)),
    "d": len(DATA_COLS),
    "colnames": DATA_COLS,
    "rownames": fit_df.GeneID.to_list()
}

meta_path = Path("../Processed_Data/bin_meta.json")
meta_dat = json.loads(meta_path.read_text(encoding="utf-8"))
meta_dat["gp_params.bin"] = meta_out

with open(meta_path, "w") as f:
    json.dump(meta_dat, f)
print("wrote gp_params.bin and updated bin_meta.json")


wrote gp_params.bin and updated bin_meta.json
